In [1]:
import time
import numpy as np
import pandas as pd
from Bio import SeqIO
from collections import Counter

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import confusion_matrix, matthews_corrcoef, roc_auc_score

from xgboost import XGBClassifier

TOTAL_START = time.time()


def stamp(msg, t0):
    print(f"[{msg}] {time.time()-t0:.1f}s")


def load_fasta(path):
    rows = []
    for r in SeqIO.parse(path, "fasta"):
        y = 1 if r.id.lower().startswith("pos") else 0
        rows.append((str(r.seq), y))
    return pd.DataFrame(rows, columns=["seq", "label"])


train_df = load_fasta("ToxicPeptidePrediction/ToxicPeptidePrediction-main/data/training_dataset.fasta")
test_df  = load_fasta("ToxicPeptidePrediction/ToxicPeptidePrediction-main/data/independent_dataset.fasta")

y_train = train_df.label.values
y_test  = test_df.label.values

In [2]:
import numpy as np
from collections import Counter

AA = "ACDEFGHIKLMNPQRSTVWY"
AA_SET = set(AA)


def clean(seq):
    seq = seq.upper()
    return "".join(c for c in seq if c in AA_SET)


def aac(seq):
    seq = clean(seq)
    L = len(seq)

    if L == 0:
        return np.zeros(len(AA), dtype=float)

    c = Counter(seq)
    return np.array([c[a] / L for a in AA], dtype=float)


def dpc(seq):
    seq = clean(seq)
    L = len(seq)

    if L < 2:
        return np.zeros(len(AA) * len(AA), dtype=float)

    pairs = (seq[i:i+2] for i in range(L - 1))
    c = Counter(pairs)

    denom = L - 1
    return np.array([c[a+b] / denom for a in AA for b in AA], dtype=float)


KD = {
    "A": 1.8, "C": 2.5, "D": -3.5, "E": -3.5, "F": 2.8,
    "G": -0.4, "H": -3.2, "I": 4.5, "K": -3.9, "L": 3.8,
    "M": 1.9, "N": -3.5, "P": -1.6, "Q": -3.5, "R": -4.5,
    "S": -0.8, "T": -0.7, "V": 4.2, "W": -0.9, "Y": -1.3
}


def phys(seq):
    seq = clean(seq)

    if len(seq) == 0:
        return np.zeros(3, dtype=float)

    vals = np.array([KD[a] for a in seq], dtype=float)

    return np.array([
        len(seq),
        vals.mean(),
        vals.std()
    ], dtype=float)


def build(df):
    return np.hstack([
        np.vstack([aac(x) for x in df.seq]),
        np.vstack([dpc(x) for x in df.seq]),
        np.vstack([phys(x) for x in df.seq])
    ])


X_train_hand = build(train_df)
X_test_hand = build(test_df)

X_train_bert = np.load("toxteller data embedding/protbert_train.npy")
X_test_bert = np.load("toxteller data embedding/protbert_test.npy")

X_train_combo = np.hstack([X_train_hand, X_train_bert])
X_test_combo = np.hstack([X_test_hand, X_test_bert])

stamp("features ready", TOTAL_START)

[features ready] 4.2s


In [3]:
def metric(y, p, thr):
    pred = (p >= thr).astype(int)

    tn, fp, fn, tp = confusion_matrix(y, pred).ravel()

    sn = tp / (tp + fn + 1e-12)
    sp = tn / (tn + fp + 1e-12)
    mcc = matthews_corrcoef(y, pred)
    auc = roc_auc_score(y, p)

    return {"Sn": sn, "Sp": sp, "MCC": mcc, "AUC": auc}


def best_threshold(y, p):
    best = None

    for thr in np.linspace(0.1, 0.9, 41):
        m = metric(y, p, thr)
        m["thr"] = thr

        if best is None or m["MCC"] > best["MCC"]:
            best = m

    return best

In [4]:
def model():
    return XGBClassifier(
        n_estimators=4000,
        max_depth=3,
        learning_rate=0.02,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="aucpr",
        early_stopping_rounds=300,
        n_jobs=-1,
        random_state=42
    )

In [16]:
import time
import numpy as np
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler


def train_model(X_train, X_test, name):

    START = time.time()

    skf = StratifiedKFold(
        n_splits=10,
        shuffle=True,
        random_state=42
    )

    oof = np.zeros(len(X_train))

    # =========================
    # OOF (10-FOLD STRATIFIED)
    # =========================
    for tr, va in skf.split(X_train, y_train):

        scaler = StandardScaler()

        X_tr = scaler.fit_transform(X_train[tr])
        X_va = scaler.transform(X_train[va])

        X1, X2, y1, y2 = train_test_split(
            X_tr,
            y_train[tr],
            test_size=0.2,
            stratify=y_train[tr],
            random_state=42
        )

        m = model()

        m.fit(
            X1,
            y1,
            eval_set=[(X2, y2)],
            verbose=False
        )

        oof[va] = m.predict_proba(X_va)[:, 1]

    # =========================
    # THRESHOLD (ONLY OOF)
    # =========================
    thr = best_threshold(y_train, oof)

    # =========================
    # FINAL MODEL (FULL TRAIN)
    # =========================
    scaler = StandardScaler()

    Xtr = scaler.fit_transform(X_train)
    Xte = scaler.transform(X_test)

    X1, X2, y1, y2 = train_test_split(
        Xtr,
        y_train,
        test_size=0.2,
        stratify=y_train,
        random_state=42
    )

    m = model()

    m.fit(
        X1,
        y1,
        eval_set=[(X2, y2)],
        verbose=False
    )

    p = m.predict_proba(Xte)[:, 1]

    # =========================
    # TEST EVALUATION
    # =========================
    test = metric(y_test, p, thr["thr"])

    print("\n" + "=" * 60)
    print(name)
    print("OOF:", thr)
    print("TEST:", test)
    print("TIME:", round(time.time() - START, 1), "sec")

    return p, thr,oof

In [18]:
#3
from sklearn.decomposition import PCA

from sklearn.preprocessing import StandardScaler

scaler_bert = StandardScaler()
X_train_bert1 = scaler_bert.fit_transform(X_train_bert)
X_test_bert1  = scaler_bert.transform(X_test_bert)
pca = PCA(n_components=256, random_state=42)
X_train_bert_pca = pca.fit_transform(X_train_bert1)
X_test_bert_pca  = pca.transform(X_test_bert1)
X_train_combo = np.hstack([X_train_hand, X_train_bert_pca])
X_test_combo = np.hstack([X_test_hand, X_test_bert_pca])
p_hand, t_hand,oof_hand = train_model(X_train_hand, X_test_hand, "HAND")
p_bert, t_bert, oof_bert = train_model(X_train_bert_pca, X_test_bert_pca, "BERT")
p_combo, t_combo, oof_combo = train_model(X_train_combo, X_test_combo, "COMBO")


HAND
OOF: {'Sn': np.float64(0.8614762386248732), 'Sp': np.float64(0.8986517898651786), 'MCC': 0.7612761733211152, 'AUC': 0.9356289712170933, 'thr': np.float64(0.48)}
TEST: {'Sn': np.float64(0.7199999999999929), 'Sp': np.float64(0.9299999999999907), 'MCC': 0.6648246687290141, 'AUC': 0.8915}
TIME: 154.5 sec

BERT
OOF: {'Sn': np.float64(0.8200202224469157), 'Sp': np.float64(0.8605299860529982), 'MCC': 0.6815781540234074, 'AUC': 0.9118494983639184, 'thr': np.float64(0.52)}
TEST: {'Sn': np.float64(0.6599999999999935), 'Sp': np.float64(0.8199999999999918), 'MCC': 0.4862645390838647, 'AUC': 0.8239}
TIME: 218.4 sec

COMBO
OOF: {'Sn': np.float64(0.8645096056622847), 'Sp': np.float64(0.9093444909344487), 'MCC': 0.7754924131455707, 'AUC': 0.9417648057032753, 'thr': np.float64(0.45999999999999996)}
TEST: {'Sn': np.float64(0.7699999999999924), 'Sp': np.float64(0.9499999999999905), 'MCC': 0.7319553114260202, 'AUC': 0.9153999999999998}
TIME: 272.4 sec


In [8]:
#1
p_hand, t_hand = train_model(X_train_hand, X_test_hand, "HAND")

p_bert, t_bert = train_model(X_train_bert, X_test_bert, "BERT")
p_combo, t_combo = train_model(X_train_combo, X_test_combo, "COMBO")


HAND
OOF: {'Sn': np.float64(0.8614762386248732), 'Sp': np.float64(0.8986517898651786), 'MCC': 0.7612761733211152, 'AUC': 0.9356289712170933, 'thr': np.float64(0.48)}
TEST: {'Sn': np.float64(0.7199999999999929), 'Sp': np.float64(0.9299999999999907), 'MCC': 0.6648246687290141, 'AUC': 0.8915}
TIME: 157.8 sec

BERT
OOF: {'Sn': np.float64(0.8185035389282099), 'Sp': np.float64(0.8763365876336584), 'MCC': 0.696868984735743, 'AUC': 0.9214450541263053, 'thr': np.float64(0.56)}
TEST: {'Sn': np.float64(0.6899999999999932), 'Sp': np.float64(0.8299999999999917), 'MCC': 0.5251721561104049, 'AUC': 0.8397}
TIME: 698.2 sec

COMBO
OOF: {'Sn': np.float64(0.8609706774519713), 'Sp': np.float64(0.9177126917712688), 'MCC': 0.7811251045421278, 'AUC': 0.9473191625782258, 'thr': np.float64(0.52)}
TEST: {'Sn': np.float64(0.7499999999999926), 'Sp': np.float64(0.9399999999999906), 'MCC': 0.7028021810549621, 'AUC': 0.9195000000000001}
TIME: 719.9 sec


In [10]:
#2
from sklearn.decomposition import PCA

from sklearn.preprocessing import StandardScaler

scaler_bert = StandardScaler()
X_train_bert1 = scaler_bert.fit_transform(X_train_bert)
X_test_bert1  = scaler_bert.transform(X_test_bert)
pca = PCA(n_components=256, random_state=42)
X_train_bert_pca = pca.fit_transform(X_train_bert1)
X_test_bert_pca  = pca.transform(X_test_bert1)
p_bert, t_bert = train_model(X_train_bert_pca, X_test_bert_pca, "BERT")
X_train_combo = np.hstack([X_train_hand, X_train_bert_pca])
X_test_combo = np.hstack([X_test_hand, X_test_bert_pca])
p_combo, t_combo = train_model(X_train_combo, X_test_combo, "COMBO")





BERT
OOF: {'Sn': np.float64(0.8200202224469157), 'Sp': np.float64(0.8605299860529982), 'MCC': 0.6815781540234074, 'AUC': 0.9118494983639184, 'thr': np.float64(0.52)}
TEST: {'Sn': np.float64(0.6599999999999935), 'Sp': np.float64(0.8199999999999918), 'MCC': 0.4862645390838647, 'AUC': 0.8239}
TIME: 222.8 sec

COMBO
OOF: {'Sn': np.float64(0.8645096056622847), 'Sp': np.float64(0.9093444909344487), 'MCC': 0.7754924131455707, 'AUC': 0.9417648057032753, 'thr': np.float64(0.45999999999999996)}
TEST: {'Sn': np.float64(0.7699999999999924), 'Sp': np.float64(0.9499999999999905), 'MCC': 0.7319553114260202, 'AUC': 0.9153999999999998}
TIME: 270.9 sec


In [20]:
import numpy as np
import time
from sklearn.metrics import confusion_matrix, matthews_corrcoef, roc_auc_score
from xgboost import XGBClassifier

TOTAL_START = time.time()

# =========================
# SAFETY CHECK
# =========================
p_hand = np.asarray(p_hand).ravel()
p_bert = np.asarray(p_bert).ravel()
p_combo = np.asarray(p_combo).ravel()

assert len(p_hand) == len(y_test)
assert len(p_bert) == len(y_test)
assert len(p_combo) == len(y_test)

In [23]:
import numpy as np
import time
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, matthews_corrcoef, roc_auc_score
from xgboost import XGBClassifier

TOTAL_START = time.time()

# =========================
# METRICS
# =========================
def metric(y_true, y_prob, thr):
    y_pred = (y_prob >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    sn = tp / (tp + fn + 1e-12)
    sp = tn / (tn + fp + 1e-12)
    mcc = matthews_corrcoef(y_true, y_pred)
    auc = roc_auc_score(y_true, y_prob)

    return {"Sn": sn, "Sp": sp, "MCC": mcc, "AUC": auc}


def best_threshold(y_true, prob):
    best = None
    for thr in np.linspace(0.1, 0.9, 81):
        m = metric(y_true, prob, thr)
        if best is None or m["MCC"] > best["MCC"]:
            best = {"thr": thr, **m}
    return best


# =========================
# MODEL
# =========================
def model():
    return XGBClassifier(
        n_estimators=600,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="aucpr",
        n_jobs=-1,
        random_state=42
    )


# =========================
# TRAIN (OOF SAFE)
# =========================
def train_model(X_train, X_test, name):
    start = time.time()

    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

    oof = np.zeros(len(X_train))
    test_pred = np.zeros(len(X_test))

    for tr, va in skf.split(X_train, y_train):

        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_train[tr])
        X_va = scaler.transform(X_train[va])
        X_te = scaler.transform(X_test)

        X1, X2, y1, y2 = train_test_split(
            X_tr,
            y_train[tr],
            test_size=0.2,
            stratify=y_train[tr],
            random_state=42
        )

        m = model()
        m.fit(X1, y1, eval_set=[(X2, y2)], verbose=False)

        oof[va] = m.predict_proba(X_va)[:, 1]
        test_pred += m.predict_proba(X_te)[:, 1] / skf.n_splits

    thr = best_threshold(y_train, oof)

    test_metrics = metric(y_test, test_pred, thr["thr"])

    print("\n" + "="*60)
    print(name)
    print("OOF:", thr)
    print("TEST:", test_metrics)
    print("TIME:", round(time.time() - start, 1), "sec")

    return oof, test_pred, thr, test_metrics


# =========================
# TRAIN 3 MODELS
# =========================
oof_hand, p_hand, t_hand, m_hand = train_model(X_train_hand, X_test_hand, "HAND")
oof_bert, p_bert, t_bert, m_bert = train_model(X_train_bert, X_test_bert, "BERT")
oof_combo, p_combo, t_combo, m_combo = train_model(X_train_combo, X_test_combo, "COMBO")


# =========================
# ENSEMBLES
# =========================
p_avg = (p_hand + p_bert + p_combo) / 3

weights = np.array([m_hand["MCC"], m_bert["MCC"], m_combo["MCC"]], dtype=float)
weights = weights / weights.sum()

p_w = (
    weights[0] * p_hand +
    weights[1] * p_bert +
    weights[2] * p_combo
)


# =========================
# STACKING (CORRECT OOF)
# =========================
meta_X_train = np.column_stack([oof_hand, oof_bert, oof_combo])
meta_X_test = np.column_stack([p_hand, p_bert, p_combo])

meta_model = XGBClassifier(
    n_estimators=150,
    max_depth=2,
    learning_rate=0.05,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

meta_model.fit(meta_X_train, y_train)
p_stack = meta_model.predict_proba(meta_X_test)[:, 1]


# =========================
# REPORT
# =========================
def report(name, p):
    thr = best_threshold(y_test, p)
    m = metric(y_test, p, thr["thr"])

    print("\n" + "="*60)
    print(name)
    print(f"Sn={m['Sn']:.3f} | Sp={m['Sp']:.3f} | MCC={m['MCC']:.3f} | AUC={m['AUC']:.3f}")
    print("thr:", round(thr["thr"], 3))


report("ENSEMBLE_AVG", p_avg)
report("ENSEMBLE_WEIGHTED", p_w)
report("ENSEMBLE_STACKING", p_stack)

print("\nTOTAL TIME:", round(time.time() - TOTAL_START, 1), "sec")


HAND
OOF: {'thr': np.float64(0.51), 'Sn': np.float64(0.8518705763397367), 'Sp': np.float64(0.9070199907019987), 'MCC': 0.7611064455887004, 'AUC': 0.9351767630828937}
TEST: {'Sn': np.float64(0.7399999999999927), 'Sp': np.float64(0.9599999999999904), 'MCC': 0.7175808220916384, 'AUC': 0.899}
TIME: 36.5 sec

BERT
OOF: {'thr': np.float64(0.5700000000000001), 'Sn': np.float64(0.7957532861476235), 'Sp': np.float64(0.8823802882380285), 'MCC': 0.6820664102251811, 'AUC': 0.9180553734031107}
TEST: {'Sn': np.float64(0.6799999999999933), 'Sp': np.float64(0.8499999999999915), 'MCC': 0.537828599567385, 'AUC': 0.8504999999999999}
TIME: 151.3 sec

COMBO
OOF: {'thr': np.float64(0.45999999999999996), 'Sn': np.float64(0.8629929221435789), 'Sp': np.float64(0.9112040911204087), 'MCC': 0.7760450822858361, 'AUC': 0.9432293113603427}
TEST: {'Sn': np.float64(0.7699999999999924), 'Sp': np.float64(0.9599999999999904), 'MCC': 0.7435443364784381, 'AUC': 0.9158999999999999}
TIME: 73.4 sec

ENSEMBLE_AVG
Sn=0.770 | S

In [ ]:
import json
import numpy as np
import time


SAVE_PATH = "toxicity_results_v1.npz"
META_PATH = "toxicity_results_meta.json"


results = {
    # predictions
    "p_hand": p_hand,
    "p_bert": p_bert,
    "p_combo": p_combo,

    # ensemble variants
    "p_avg": (p_hand + p_bert + p_combo) / 3,

    # thresholds
    "thr_hand": t_hand["thr"],
    "thr_bert": t_bert["thr"],
    "thr_combo": t_combo["thr"],

    # oof thresholds (full dict -> json later)
}


# save numeric arrays (fast + compact)
np.savez_compressed(
    SAVE_PATH,
    **{
        k: v for k, v in results.items()
        if isinstance(v, np.ndarray)
    }
)


# save metadata (metrics + thresholds + runtime info)
meta = {
    "hand": t_hand,
    "bert": t_bert,
    "combo": t_combo,
    "timestamp": time.time(),
    "note": "NO LEAKAGE OOF + TEST ONLY EVAL"
}

with open(META_PATH, "w") as f:
    json.dump(meta, f, indent=4)

print("Saved to:")
print(" ->", SAVE_PATH)
print(" ->", META_PATH)